In [2]:
import cv2
import os
import numpy as np
from scipy.ndimage import label


coordinates_folder = '/playpen-raid2/qinliu/data/ecDNA/coords'
masks_folder = '/playpen-raid2/qinliu/projects/LabelEngine/saves/model_0424_2024/inter/inter_unet_2048x2448_ecDNA/186/vis/val'

mask_files = sorted(os.listdir(masks_folder))
sum_dev = 0
for file in mask_files:
    mask = cv2.imread(os.path.join(masks_folder, file))[:, :, 0]
    mask_cc, num_cc = label(mask)

    image_barename = file[11:].split('.')[0]
    num_gt = len(np.load(os.path.join(coordinates_folder, image_barename+'.npy')))
    sum_dev += abs(1 - num_cc / (num_gt))

acc = 1 - sum_dev / len(mask_files)
print(acc)

/home/qinliu19/.local/lib/python3.8/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


0.8200033684922199


In [72]:
import matplotlib.pyplot as plt

acc_mean_prev = 0
acc_mean_update = 0
for idx, file in enumerate(mask_files):
    mask = cv2.imread(os.path.join(masks_folder, file))[:, :, 0].astype(np.int32)

    mask_cc, num_cc = label(mask)
    assert num_cc == len(np.unique(mask_cc)) - 1

    image_barename = file[11:].split('.')[0]
    gt_coords = np.load(os.path.join(coordinates_folder, image_barename+'.npy'))
    num_gt_coords = len(gt_coords)

    duplicates = [0 for _ in range(num_cc)]
    for i in range(gt_coords.shape[0]):
        coords = gt_coords[i]
        x, y = round(coords[0]), round(coords[1])

        cc_idx = mask_cc[y, x]
        if cc_idx > 0:
            duplicates[cc_idx - 1] += 1

    duplicates_count = 0
    for elm in duplicates:
        if elm > 2:
            duplicates_count += (elm - 1)

    acc_prev = 1 - abs(1 - num_cc / num_gt_coords)
    acc_update = 1 - abs(1 - (num_cc + duplicates_count) / num_gt_coords)
    acc_mean_prev += acc_prev
    acc_mean_update += acc_update

    print(idx, file, acc_prev, acc_update, duplicates_count)

acc_mean_prev /= len(mask_files)
acc_mean_update /= len(mask_files)

print(acc_mean_prev, acc_mean_update)

0 000000_seg_NCIH2170_Antibiotics_DAPI_Blank_Control_101_Merge.png 0.9774436090225563 0.9774436090225563 0
1 000001_seg_NCIH2170_Antibiotics_DAPI_Blank_Control_109_Merge.png 0.9505494505494505 0.9505494505494505 0
2 000002_seg_NCIH2170_Antibiotics_DAPI_Blank_Control_113_Merge.png 0.9648093841642229 0.9706744868035191 2
3 000003_seg_NCIH2170_Antibiotics_DAPI_Blank_Control_18_Merge.png 0.9872611464968153 0.9872611464968153 0
4 000004_seg_NCIH2170_Antibiotics_DAPI_Blank_Control_1_Merge.png 0.9531914893617022 0.9531914893617022 0
5 000005_seg_NCIH2170_Antibiotics_DAPI_Blank_Control_20_Merge.png 0.806201550387597 0.806201550387597 0
6 000006_seg_NCIH2170_Antibiotics_DAPI_Blank_Control_22_Merge.png 0.9973474801061007 0.9973474801061007 0
7 000007_seg_NCIH2170_Antibiotics_DAPI_Blank_Control_24_Merge.png 0.9869281045751634 0.9869281045751634 0
8 000008_seg_NCIH2170_Antibiotics_DAPI_Blank_Control_27_Merge.png 0.8930817610062893 0.8930817610062893 0
9 000009_seg_NCIH2170_Antibiotics_DAPI_Blank_C

In [ ]:
import matplotlib.pyplot as plt

save_folder = '/playpen-raid2/qinliu/projects/LabelEngine/saves/val_results'
for file in mask_files:
    mask = cv2.imread(os.path.join(masks_folder, file))[:, :, 0].astype(np.int32)

    plt.figure(figsize=(50, 40))
    # plt.imshow(mask, cmap='gray')

    mask_cc, num_cc = label(mask)
    assert num_cc == len(np.unique(mask_cc)) - 1

    image_barename = file[11:].split('.')[0]
    gt_coords = np.load(os.path.join(coordinates_folder, image_barename+'.npy'))
    num_gt_coords = len(gt_coords)

    points = []
    catched_points = []
    num_catches = 0
    for i in range(gt_coords.shape[0]):
        coords = gt_coords[i]
        x, y = round(coords[0]), round(coords[1])
        points.append((x, y))
        if mask_cc[y, x] > 0:
            catched_points.append((x, y))

    x_coords, y_coords = zip(*points)
    plt.scatter(x_coords, y_coords, color='red', marker='x')
    plt.tight_layout()
    plt.savefig(os.path.join(save_folder, f'{image_barename}_gt_{num_gt_coords}_pred_{num_cc}.png'), bbox_inches='tight', pad_inches=0)

    acc = 1 - abs(1 - num_cc / num_gt_coords)
    print(num_cc, num_gt_coords, acc)